# Ridge Regresyonu (L2 Regularization)

## Önce Temel Kavramlar

Regresyon probleminde elde bulunanlar:

- Bir grup **gözlem** (örnek, satır) — buna `n` denir
- Her gözlem için bir grup **özellik** (sütun, değişken) — buna `p` denir
- Tahmin edilmeye çalışılan bir **hedef değer** — buna `y` denir

Örnek: 100 evin fiyatının tahmin edilmesi.
- `n = 100` (100 ev)
- `p = 3` (metrekare, oda sayısı, yaş — 3 özellik)
- `y` = evin fiyatı

Amaç aşağıdaki formülün bulunmasıdır:

$$\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \dots + \beta_p x_p$$

Burada $\hat{\beta}$ (beta şapka) **tahmin edilen katsayılar** anlamına gelir.
Şapka sembolü ($\hat{\phantom{x}}$) her zaman "tahmin edilen" anlamına gelir — gerçek değil, model tarafından bulunan değer.

---

## OLS Nedir ve Neden Yetersiz Kalabilir?

**OLS (Ordinary Least Squares — Sıradan En Küçük Kareler)**, katsayıların bulunması için
tahmin hatalarının karelerini minimize eder:

$$\min_{\beta} \sum_{i=1}^{n}(y_i - \hat{y}_i)^2$$

Türkçesi: "Her gözlem için (gerçek değer − tahmin)² toplamının en küçük yapılması."

Matris formundaki çözüm:

$$\hat{\beta}_{OLS} = (X^T X)^{-1} X^T y$$

Buradaki semboller:
- $X$ → Tüm veriyi tutan matris (satırlar = gözlemler, sütunlar = özellikler)
- $X^T$ → X matrisinin **transpozu** (satır ve sütunları yer değiştirilmiş hali)
- $(...)^{-1}$ → Matrisin **tersi** (sayılardaki "1/x" işleminin matris karşılığı)
- $y$ → Gerçek hedef değerlerin vektörü

**Sorun:** Bu formülün çalışabilmesi için $X^T X$ matrisinin tersinin alınabilmesi gerekir.
İki durumda bu mümkün olmayabilir:

1. Özelliklerin birbiriyle çok yüksek korelasyonlu olması (**multicollinearity**)
2. Özellik sayısının gözlem sayısından fazla olması ($p > n$)

Bu durumlarda katsayılar astronomik büyüklüklere ulaşabilir ve model eğitim verisini
ezberleyebilir — yeni veride başarısız olabilir. Buna **overfitting** denir.

####

## Ridge Regresyonu Nedir?

Ridge, OLS'ye bir **ceza terimi** ekler. Katsayılar çok büyüdüğünde cezalandırılır:

$$\min_{\beta} \underbrace{\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}_{\text{OLS'den gelen hata}} + \underbrace{\lambda \sum_{j=1}^{p} \beta_j^2}_{\text{Ceza terimi (L2)}}$$

Yeni sembollerin anlamı:
- $\lambda$ (lambda) → Cezanın şiddeti. Kullanıcı tarafından ayarlanır. 0 ise Ridge = OLS olur.
- $\sum_{j=1}^{p} \beta_j^2$ → Tüm katsayıların karelerinin toplamı
- $j$ → özellik indisi (1'den p'ye kadar)
- $i$ → gözlem indisi (1'den n'e kadar)

**Sezgisel açıklama:**
Model aynı anda iki hedefi gerçekleştirmeye çalışır:
1. Tahmin hatasının küçük tutulması (veriye iyi uyum sağlanması)
2. Katsayıların küçük tutulması (çok büyümemesi)

$\lambda$ bu iki hedef arasındaki dengeyi kurar.

| $\lambda$ | Ne olur? |
|---|---|
| 0 | Ceza yok, Ridge = OLS |
| Küçük (0.01) | Hafif ceza uygulanır, katsayılar biraz küçülür |
| Büyük (100) | Güçlü ceza uygulanır, katsayılar sıfıra yaklaşır |
| $\to \infty$ | Tüm katsayılar sıfıra yaklaşır (ama tam sıfır olmaz) |

> **Kritik fark:** Ridge katsayıları sıfıra *yaklaştırır* ancak hiçbirini tam sıfır yapmaz.
> LASSO (L1) ise bazı katsayıları tam sıfır yapabilir — özellik elemesi gerçekleştirebilir.
> Ridge özellik elemesi yapmaz, tüm özellikleri modelde tutar.

<div style="display: flex; gap: 20px;">
  <img src="../figures/lasso-l1.png" width="45%"/>
  <img src="../figures/ridge-l2.png" width="45%"/>
</div>

Görseli okuma rehberi:
- Elips şeklindeki çizgiler = hata yüzeyi (kırmızı yıldıza yaklaşıldıkça hata azalır)
- Kırmızı yıldız = OLS çözümü (ceza olmadan en iyi nokta)
- Mavi daire (Ridge) = L2 kısıtı: $\beta_1^2 + \beta_2^2 \leq t$
- Kırmızı elmas (LASSO) = L1 kısıtı: $|\beta_1| + |\beta_2| \leq t$
- Mavi/kırmızı nokta = Regularization çözümü (elipslerin kısıt bölgesine ilk temas ettiği nokta)

Ridge'de daire köşesiz olduğu için çözüm eksenlere denk gelmez → hiçbir katsayı sıfır olmaz.
LASSO'da elmas köşeli olduğu için çözüm çoğunlukla bir eksene denk gelir → katsayı sıfır olur.

## Matris Notasyonu ve Kapalı Form Çözümü

Toplamlar yerine matris yazımı hem daha kompakt hem de bilgisayarlar için daha hızlıdır:

$$\min_{\beta} \| y - X\beta \|^2 + \lambda \|\beta\|^2$$

Burada:
- $\| v \|^2$ → bir vektörün **L2 normu karesi** = elemanların karelerinin toplamı
  Örnek: $v = [3, 4]$ ise $\|v\|^2 = 3^2 + 4^2 = 25$
- $\| y - X\beta \|^2$ → tahmin hatalarının kareler toplamı (OLS'deki hata terimi)
- $\|\beta\|^2 = \beta^T \beta$ → katsayıların kareler toplamı (ceza terimi)

### Kapalı Form Çözümünün Türetilmesi

$\beta$'ya göre türev alınır ve sıfıra eşitlenir (minimumun bulunması için):

**Adım 1 — Açılım:**

$$J(\beta) = y^Ty - 2\beta^T X^T y + \beta^T X^T X \beta + \lambda \beta^T \beta$$

**Adım 2 — $\beta$'ya göre türev:**

$$\frac{\partial J}{\partial \beta} = -2X^T y + 2X^T X \beta + 2\lambda \beta$$

**Adım 3 — Sıfıra eşitle ve sadeleştir:**

$$0 = -2X^T y + 2(X^T X + \lambda I)\beta$$

**Adım 4 — $\beta$'yı yalnız bırak:**

$$\boxed{\hat{\beta}_{Ridge} = (X^T X + \lambda I)^{-1} X^T y}$$

Burada $I$ → **birim matris (identity matrix)**. Köşegeni 1, geri kalanı 0 olan kare matris:

$$I = \begin{bmatrix} 1 & 0 & 0 \\ 0 & 1 & 0 \\ 0 & 0 & 1 \end{bmatrix}$$

### Neden $\lambda I$ Eklenmesi Matrisi Her Zaman Tersinir Yapar?

$X^T X$ matrisinin bazı köşegen elemanları sıfır olabilir → tersi alınamayabilir.
$\lambda I$ eklendiğinde köşegendeki her eleman en az $\lambda$ kadar büyür:

$$\text{eigenvalue}(X^T X + \lambda I) = \mu_i + \lambda$$

$\lambda > 0$ ise tüm değerler pozitif olur → matrisin tersi her zaman alınabilir.

Özetle: $\lambda I$ eklenmesi, matematiksel kararsızlığın giderilmesi için garantili bir yöntemdir.

## Adım Adım Elle Hesaplama Örneği

Küçük bir veri seti üzerinden tüm adımların elle izlenmesi.

### Veri

| Gözlem | $x_1$ | $x_2$ | $y$ |
|---|---|---|---|
| 1 | 1 | 2 | 5 |
| 2 | 2 | 3 | 8 |
| 3 | 3 | 1 | 6 |

- $n = 3$ gözlem, $p = 2$ özellik, $\lambda = 1$
- Şimdilik intercept eklenmiyor — yalnızca $\beta_1$ ve $\beta_2$ hesaplanıyor

$$X = \begin{bmatrix} 1 & 2 \\ 2 & 3 \\ 3 & 1 \end{bmatrix}, \quad y = \begin{bmatrix} 5 \\ 8 \\ 6 \end{bmatrix}$$

---

### Adım 1: $X^T$ (Transpoz) Hesaplama

Transpoz alma işlemi = satırların sütun, sütunların satır hâline getirilmesi:

$$X^T = \begin{bmatrix} 1 & 2 & 3 \\ 2 & 3 & 1 \end{bmatrix}$$

---

### Adım 2: $X^T X$ Hesaplama

$(2 \times 3) \cdot (3 \times 2) = (2 \times 2)$ boyutunda matris elde edilir.

$$X^T X = \begin{bmatrix} 1 & 2 & 3 \\ 2 & 3 & 1 \end{bmatrix} \begin{bmatrix} 1 & 2 \\ 2 & 3 \\ 3 & 1 \end{bmatrix}$$

Eleman eleman hesaplama:
- $(1,1)$: $1\cdot1 + 2\cdot2 + 3\cdot3 = 1+4+9 = 14$
- $(1,2)$: $1\cdot2 + 2\cdot3 + 3\cdot1 = 2+6+3 = 11$
- $(2,1)$: $2\cdot1 + 3\cdot2 + 1\cdot3 = 2+6+3 = 11$
- $(2,2)$: $2\cdot2 + 3\cdot3 + 1\cdot1 = 4+9+1 = 14$

$$X^T X = \begin{bmatrix} 14 & 11 \\ 11 & 14 \end{bmatrix}$$

Bu matris her zaman **simetrik** olur — yani $X^TX_{ij} = X^TX_{ji}$.

---

### Adım 3: $\lambda I$ Ekleme

$\lambda = 1$, $I = \begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix}$

$$X^T X + \lambda I = \begin{bmatrix} 14 & 11 \\ 11 & 14 \end{bmatrix} + \begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix} = \begin{bmatrix} 15 & 11 \\ 11 & 15 \end{bmatrix}$$

Yalnızca köşegen elemanlar 1 artmıştır. Köşegen dışındaki elemanlar değişmemiştir.

---

### Adım 4: $X^T y$ Hesaplama

$$X^T y = \begin{bmatrix} 1 & 2 & 3 \\ 2 & 3 & 1 \end{bmatrix} \begin{bmatrix} 5 \\ 8 \\ 6 \end{bmatrix}$$

- İlk eleman: $1\cdot5 + 2\cdot8 + 3\cdot6 = 5+16+18 = 39$
- İkinci eleman: $2\cdot5 + 3\cdot8 + 1\cdot6 = 10+24+6 = 40$

$$X^T y = \begin{bmatrix} 39 \\ 40 \end{bmatrix}$$

---

### Adım 5: $(X^T X + \lambda I)^{-1}$ Hesaplama

$2\times2$ matrisin tersi için kullanılan formül:

$$\begin{bmatrix} a & b \\ c & d \end{bmatrix}^{-1} = \frac{1}{ad - bc}\begin{bmatrix} d & -b \\ -c & a \end{bmatrix}$$

Determinant:

$$ad - bc = 15 \cdot 15 - 11 \cdot 11 = 225 - 121 = 104$$

$$\left(\begin{bmatrix} 15 & 11 \\ 11 & 15 \end{bmatrix}\right)^{-1} = \frac{1}{104}\begin{bmatrix} 15 & -11 \\ -11 & 15 \end{bmatrix}$$

---

### Adım 6: $\hat{\beta}_{Ridge}$ Hesaplama

$$\hat{\beta}_{Ridge} = \frac{1}{104}\begin{bmatrix} 15 & -11 \\ -11 & 15 \end{bmatrix} \begin{bmatrix} 39 \\ 40 \end{bmatrix}$$

- $\hat{\beta}_1$: $(15 \cdot 39 + (-11) \cdot 40) / 104 = (585 - 440)/104 = 145/104 \approx 1.394$
- $\hat{\beta}_2$: $((-11) \cdot 39 + 15 \cdot 40) / 104 = (-429 + 600)/104 = 171/104 \approx 1.644$

$$\hat{\beta}_{Ridge} = \begin{bmatrix} 1.394 \\ 1.644 \end{bmatrix}$$

---

### OLS ile Karşılaştırma ($\lambda = 0$)

$\lambda = 0$ olması durumunda:
- Determinant: $14 \cdot 14 - 11 \cdot 11 = 196 - 121 = 75$

$$\hat{\beta}_{OLS} = \frac{1}{75}\begin{bmatrix} 14 & -11 \\ -11 & 14 \end{bmatrix} \begin{bmatrix} 39 \\ 40 \end{bmatrix} = \begin{bmatrix} 1.413 \\ 1.747 \end{bmatrix}$$

| | $\hat{\beta}_1$ | $\hat{\beta}_2$ |
|---|---|---|
| OLS ($\lambda=0$) | 1.413 | 1.747 |
| Ridge ($\lambda=1$) | 1.394 | 1.644 |

Ridge her iki katsayıyı da sıfıra doğru **büzmüştür (shrinkage)** — ancak sıfır yapmamıştır.

In [8]:
import numpy as np

# --- Veri ---
X = np.array([
    [1, 2],
    [2, 3],
    [3, 1],
], dtype=float)

y = np.array([5, 8, 6], dtype=float)
lam = 1.0

# --- Adım 1: X^T ---
Xt = X.T
print("X^T:\n", Xt)

# --- Adım 2: X^T X ---
XtX = Xt @ X
print("\nX^T X:\n", XtX)

# --- Adım 3: X^T X + lambda * I ---
I = np.eye(X.shape[1])          # 2x2 birim matris
A = XtX + lam * I
print("\nX^T X + λI:\n", A)

# --- Adım 4: X^T y ---
Xty = Xt @ y
print("\nX^T y:\n", Xty)

# --- Adım 5: Ters ---
A_inv = np.linalg.inv(A)
print("\n(X^T X + λI)^{-1}:\n", np.round(A_inv, 6))

# --- Adım 6: Beta ---
beta_ridge = A_inv @ Xty
print("\nRidge katsayıları:", np.round(beta_ridge, 4))

# --- OLS (lambda=0) ---
beta_ols = np.linalg.inv(XtX) @ Xty
print("OLS katsayıları  :", np.round(beta_ols, 4))

print("\nShrinkage (Ridge/OLS):")
print(np.round(beta_ridge / beta_ols * 100, 1), "%")

X^T:
 [[1. 2. 3.]
 [2. 3. 1.]]

X^T X:
 [[14. 11.]
 [11. 14.]]

X^T X + λI:
 [[15. 11.]
 [11. 15.]]

X^T y:
 [39. 40.]

(X^T X + λI)^{-1}:
 [[ 0.144231 -0.105769]
 [-0.105769  0.144231]]

Ridge katsayıları: [1.3942 1.6442]
OLS katsayıları  : [1.4133 1.7467]

Shrinkage (Ridge/OLS):
[98.6 94.1] %


## Özellik Ölçeklendirme (Standardization)

Ridge ceza terimi $\lambda \sum \beta_j^2$ tüm katsayılara eşit ceza uygular.
Ancak özellikler farklı ölçeklerdeyse bu adil olmaz:

- $x_1$: yaş → [20, 60]
- $x_2$: maaş → [20000, 80000]

Maaşı tahmin eden $\beta_2$ zaten çok küçük olmak zorundadır (büyük sayılarla çarpıldığı için).
Ridge bu küçük $\beta_2$'yi daha az cezalandırır, büyük $\beta_1$'i daha fazla cezalandırır.
Ceza özelliğin **önemine** göre değil **ölçeğine** göre dağılmış olur. Bu istenen bir durum değildir.

**Çözüm:** Her özelliğin standart normal dağılıma çekilmesi.

$$x_j^{scaled} = \frac{x_j - \mu_j}{\sigma_j}$$

- $\mu_j$ → ilgili özelliğin **ortalaması**
- $\sigma_j$ → ilgili özelliğin **standart sapması**

Sonuç: Her özelliğin ortalaması 0, standart sapması 1 olur.
Böylece tüm özellikler aynı ölçekte bulunur → ceza adil şekilde dağılır.

**Kritik kural:** $\mu$ ve $\sigma$ yalnızca **eğitim verisinden** hesaplanmalıdır.
Test verisine de aynı $\mu$ ve $\sigma$ uygulanmalıdır.
Aksi durumda **data leakage** oluşur — test verisi modele sızdırılmış olur.

### Intercept Cezalandırılmaz

$\beta_0$ (intercept / sabit terim) ceza dışında tutulur.
Bunun nedeni intercept'in modelin veri merkezine oturmasını sağlamasıdır.
Cezalandırılması durumunda model veri merkezinden uzaklaşabilir.

Uygulamada: $\lambda I$ matrisinin intercept'e karşılık gelen köşegen elemanı 0 bırakılır.

In [9]:
import numpy as np

def standardize(X_train, X_test=None):
    mu = X_train.mean(axis=0)
    sigma = X_train.std(axis=0)
    sigma[sigma == 0] = 1           # sabit sütun koruması

    X_train_scaled = (X_train - mu) / sigma

    if X_test is not None:
        X_test_scaled = (X_test - mu) / sigma
        return X_train_scaled, X_test_scaled, mu, sigma

    return X_train_scaled, mu, sigma


# --- Örnek ---
X = np.array([
    [25,  30000],
    [35,  55000],
    [45,  80000],
    [55,  47000],
    [30,  62000],
], dtype=float)

X_scaled, mu, sigma = standardize(X)

print("Orijinal X:\n", X)
print("\nStandardize X:\n", np.round(X_scaled, 4))
print("\nOrtalama (mu)    :", np.round(mu, 2))
print("Std sapma (sigma):", np.round(sigma, 2))
print("\nKontrol - ortalama:", np.round(X_scaled.mean(axis=0), 10))
print("Kontrol - std     :", np.round(X_scaled.std(axis=0), 10))

Orijinal X:
 [[2.5e+01 3.0e+04]
 [3.5e+01 5.5e+04]
 [4.5e+01 8.0e+04]
 [5.5e+01 4.7e+04]
 [3.0e+01 6.2e+04]]

Standardize X:
 [[-1.207  -1.5022]
 [-0.2785  0.0121]
 [ 0.6499  1.5264]
 [ 1.5784 -0.4725]
 [-0.7428  0.4361]]

Ortalama (mu)    : [3.80e+01 5.48e+04]
Std sapma (sigma): [1.077000e+01 1.650939e+04]

Kontrol - ortalama: [ 0. -0.]
Kontrol - std     : [1. 1.]


## Bias-Variance Tradeoff

Ridge, OLS'ye göre **biased** (yanlı) bir tahmindir:

$$E[\hat{\beta}_{Ridge}] \neq \beta_{gercek}$$

Yani ortalamada tam doğru cevap bulunmaz — bilinçli olarak bir miktar sapma oluşturulur.
Buna karşılık **varyans** (veriye duyarlılık) azaltılır.

<img src="../figures/trade-off.png" width="70%"/>

Grafiği okuma rehberi:
- Mavi çizgi = Eğitim hatası: λ arttıkça artar (model veriye daha az uyum sağlar)
- Kırmızı çizgi = Test hatası: önce düşer, sonra tekrar artar (U şekli)
- Yeşil nokta = Optimal λ: test hatasının en düşük olduğu nokta
- Sol taraf (küçük λ) = Overfitting bölgesi: eğitim performansı iyi, test performansı kötü
- Sağ taraf (büyük λ) = Underfitting bölgesi: her iki performans da kötü

**Temel fikir:** Az miktarda bias eklenerek varyansın büyük ölçüde azaltılması durumunda,
toplam hata azaltılabilir. Optimal λ bu dengenin kurulduğu noktadır.

## Regularization Path

λ arttıkça katsayıların davranışı nasıl değişir?

<img src="../figures/coefficient-shrinkage.png" width="70%"/>

Grafiği okuma rehberi:
- Her renkli çizgi = bir katsayı ($\beta_1, \beta_2, \dots$)
- X ekseni = λ değeri (logaritmik ölçek)
- Y ekseni = katsayının değeri
- λ küçükken katsayılar OLS değerlerine yakın olur (büyük değerler), λ büyüdükçe sıfıra yaklaşır
- Hiçbir çizgi tam sıfıra ulaşmaz — bu Ridge'in temel özelliğidir

## λ'nın Regresyon Doğrusuna Etkisi

<img src="../figures/reg-params-lambda.png" width="70%"/>

- λ = 0 (kırmızı): OLS çözümü, en dik eğim
- λ = 0.1 (sarı): Neredeyse aynı eğim
- λ = 1 (yeşil): Hafif düzleşme
- λ = 10 (mavi): Belirgin düzleşme
- λ = 100 (mor): Çok düz, katsayılar önemli ölçüde küçülmüş

Tüm doğrular veri noktaları arasından geçmeye çalışır,
ancak büyük λ değerleri bunu sınırlar — sonuçta daha basit ve daha genellenebilir bir model elde edilir.

## Avantajlar ve Dezavantajlar

| | Açıklama |
|---|---|
| **Avantaj** | Multicollinearity problemini çözer — $X^TX + \lambda I$ her zaman tersinirdir |
| **Avantaj** | Kapalı form çözümü bulunur, iterasyona ihtiyaç duyulmaz |
| **Avantaj** | Bias-variance dengesinin optimize edilmesini sağlar |
| **Avantaj** | Tüm özelliklerin modelde tutulmasını sağlar |
| **Dezavantaj** | Özellik seçimi gerçekleştirmez (LASSO gerçekleştirebilir) |
| **Dezavantaj** | λ seçimi kritiktir, cross-validation gerektirir |
| **Dezavantaj** | Özellik ölçeklendirilmesi zorunludur |
| **Dezavantaj** | Outlier'lara karşı OLS kadar duyarlıdır |

## Yöntemler Arası Karşılaştırma

| | OLS | Ridge (L2) | LASSO (L1) | Elastic Net |
|---|---|---|---|---|
| Ceza | Yok | $\lambda \sum \beta_j^2$ | $\lambda \sum |\beta_j|$ | İkisi birden |
| Özellik seçimi | Hayır | Hayır | Evet | Kısmen |
| Katsayı = 0 olur mu? | — | Hayır | Evet | Evet |
| Kapalı form çözümü var mı? | Evet | Evet | Hayır | Hayır |
| Multicollinearity ile başa çıkma | Zayıf | Güçlü | Orta | Güçlü |